In [1]:
import requests 
response = requests.get("https://string-db.org/api/image/network?identifiers=PTCH1%0dSHH%0dGLI1%0dSMO%0dGLI3")
with open('string_network.png', 'wb') as fh:
    fh.write(response.content)

In [4]:
string_api_url = "https://version-12-0.string-db.org/api"
import pandas as pd
from io import StringIO
def get_string_interactions(genes, species=9606, required_score=400, add_nodes=0): 
    """
    Query StringDB for interactions among a list of genes. 
    Args: 
        genes: list of gene symbols, e.g. ["COL5A1", "TNXB", "FBN1"] 
        species: NCBI taxon ID (9606 = human) 
        required score: confidence threshold (0-1000) 
        add_nodes: number of additional interaction partners to include

    Returns: 
        list of dicts, one per interaction 
    """
    request_url = f"{string_api_url}/tsv/network" 

    params = {

    "identifiers" : "%0d".join(genes),
    "species" : 9606,
    "required_score" : required_score, 
    "add_nodes" : add_nodes, 
    "caller_identity" : "heds_pathway_agent"
    } 

    response = requests.post(request_url, data=params) 
    response.raise_for_status() 

    df = pd.read_csv(StringIO(response.text), sep="\t") 
    return df

In [6]:
if __name__ == "__main__":
    test_genes = ["ACKR2", "APOE", "PPARG", "APOC1", "MMP1"]
    df = get_string_interactions(test_genes) 
    #print(df)
    print(df[["preferredName_A", "preferredName_B", "score"]])

  preferredName_A preferredName_B  score
0            APOE           PPARG  0.740
1            APOE           APOC1  0.999


In [7]:
kegg_api_url = "https://rest.kegg.jp"
def get_kegg_id(symbol, org = "hsa"):
    """
    Retrieve KEGG ID for a given gene symbol
    Args: 
        symbol: gene symbol as string e.g. "COL5A1"
        org: KEGG organism code (default "hsa" = human) 
    """
    
    request_url = f"{kegg_api_url}/find/{org}/{symbol}"
    id_response = requests.get(request_url) 

    if id_response.status_code != 200 or not id_response.text.strip(): 
        return None
    
    for line in id_response.text.strip().split("\n"): 
        parts = line.split("\t") 
        kegg_id = parts[0]
        aliases = parts[1].split(";")[0].split(", ") 

        if symbol.lower() in [a.lower() for a in aliases]: 
            return kegg_id 

In [8]:
if __name__ == "__main__":
    test_gene = "KRAS" 
    kegg_id = get_kegg_id(test_gene)
    print(kegg_id)

hsa:4893


In [9]:
def get_kegg_pathways(genes, org="hsa"):
    """
    For each gene, find KEGG pathway it's linked to. 

    Args: 
        genes: list of gene symbols 
        org: KEGG organism code (default "hsa" = human) 

    Returns: 
        Dataframe with columns: gene, kegg_gene_id, pathway_id
    """

    # Get KEGG id for each gene symbol
    gene_to_kegg_id = {}
    for gene in genes: 
        kegg_id = get_kegg_id(gene, org=org)
        if kegg_id is None: 
            print(f"Warning: No KEGG ID found for {gene}") 
        gene_to_kegg_id[gene] = kegg_id # Add to gene symbol, KEGG id dictionary]

        # For each gene, get linked pathways
    records = []
    for gene, kegg_id in gene_to_kegg_id.items(): 
        if kegg_id is None: 
            continue 
        
        find_url = f"{kegg_api_url}/link/pathway/{kegg_id}" 
        link_response = requests.get(find_url)
        link_response.raise_for_status()

        if not link_response.text.strip(): 
            continue

        for line in link_response.text.strip().split("\n"): 
            _, pathway_id = line.split("\t")
            records.append({
                "gene": gene, 
                "kegg_gene_id": kegg_id, 
                "pathway_id": pathway_id
            }) 

    return pd.DataFrame(records) 
        

In [10]:
if __name__ == "__main__":
    test_genes = ["ACKR2", "APOE", "PPARG", "APOC1", "KRAS"]
    kegg_df = get_kegg_pathways(test_genes)
    print(kegg_df)

     gene kegg_gene_id     pathway_id
0    APOE      hsa:348  path:hsa04979
1    APOE      hsa:348  path:hsa05010
2   PPARG     hsa:5468  path:hsa03320
3   PPARG     hsa:5468  path:hsa04148
4   PPARG     hsa:5468  path:hsa04152
..    ...          ...            ...
91   KRAS     hsa:4893  path:hsa05226
92   KRAS     hsa:4893  path:hsa05230
93   KRAS     hsa:4893  path:hsa05231
94   KRAS     hsa:4893  path:hsa05235
95   KRAS     hsa:4893  path:hsa05417

[96 rows x 3 columns]


In [11]:
open_targets_url = "https://api.platform.opentargets.org/api/v4/graphql"
def get_ensembl_id(gene_symbol, species="homo_sapiens"):
    """
    Retrieve ensemble gene ID for a given gene symbol from Open Targets Search
    Args: 
        gene symbol: e.g. "COL5A1"
        species: organism (default = homo_sapiens) 

    Returns: 
        Dataframe with columns: gene, kegg_gene_id, pathway_id
    """
    query_string = """
    query search($queryString: String!) { 
        search(queryString: $queryString, entityNames: ["target"]) {
            hits { 
                id
                name
                entity
            }
        }
    }
    """
    variables = {"queryString": gene_symbol}
    response = requests.post(open_targets_url, json={"query": query_string, "variables": variables})
    response.raise_for_status()
    data = response.json()

    hits = data["data"]["search"]["hits"] 
    for hit in hits: 
        if hit["name"].upper() == gene_symbol.upper(): 
            return hit["id"]
    return None 

In [12]:
def get_disease_associations(gene_symbol, size = 10): 
    """
    Get disease associations for a gene from Open Targets 

    Args: 
        gene_symbol: Gene symbol, e.g. "COL5A1"
        size: max number of associated diseases to return
    Returns: 
        DataFrame with columns: diseases_name, overall_score
    """
    
    gene_id = get_ensembl_id(gene_symbol)

    # Check if ensembl id was returned
    if gene_id is None:
        print(f"Warning: could not resolve Ensembl ID for {gene_symbol}, skipping")
        return pd.DataFrame(columns=["disease_name", "overall_score"])
        
    query_string = """
    query target($ensemblId: String!, $size: Int!){
        target(ensemblId: $ensemblId) {
            approvedSymbol
            associatedDiseases(page: {index: 0, size: $size}) {
                rows { 
                    disease {
                        name
                    }
                    score
                }
            }
        }
    }
    """
    variables = {"ensemblId": gene_id, "size": size} 
    response = requests.post(open_targets_url, json={"query": query_string, "variables": variables})
    response.raise_for_status()
    data = response.json()

    rows = data["data"]["target"]["associatedDiseases"]["rows"] 
    records = [{"disease_name": r["disease"]["name"], "overall_score": r["score"]} for r in rows]
    return pd.DataFrame(records) 

In [13]:
if __name__ == "__main__":
    symbol = "ACKR3"
    ensembl_id = get_ensembl_id(symbol)
    print(f"{symbol} -> {ensembl_id}")
    
    if ensembl_id:
        df = get_disease_associations(symbol)
        print(df)

ACKR3 -> ENSG00000144476
                              disease_name  overall_score
0           oculomotor-abducens synkinesis       0.498933
1                      pancreatic neoplasm       0.380144
2                       cutaneous melanoma       0.375031
3                     colon adenocarcinoma       0.372490
4                colorectal adenocarcinoma       0.369949
5  endometrial endometrioid adenocarcinoma       0.369580
6           male reproductive organ cancer       0.317430
7                         breast carcinoma       0.305937
8                 hepatocellular carcinoma       0.305791
9                           lung carcinoma       0.304848


In [14]:
def get_disease_id(disease_name):
    """
    Resolve a disease name to its Open Targets disease ID.
    """
    query_string = """
    query search($queryString: String!) {
        search(queryString: $queryString, entityNames: ["disease"]) {
            hits {
                id
                name
                entity
            }
        }
    }
    """
    variables = {"queryString": disease_name}
    response = requests.post(open_targets_url, json={"query": query_string, "variables": variables})
    response.raise_for_status()
    data = response.json()
    hits = data["data"]["search"]["hits"]
    for hit in hits:
        print(hit)  # print all hits so you can see what's actually available
    return hits[0]["id"] if hits else None

In [15]:
if __name__ == "__main__":
    disease_id = get_disease_id("hypermobile Ehlers-Danlos syndrome")

{'id': 'MONDO_0007523', 'name': 'Ehlers-Danlos syndrome, hypermobility type', 'entity': 'disease'}


In [16]:
def get_disease_target_score(disease_id, gene_symbol):
    """
    Get the association score between a specific disease and a specific gene.
    
    Args:
        disease_id: Open Targets disease ID, e.g. "MONDO_0007523"
        gene_id: Ensembl gene ID, e.g. "ENSG00000130635"
    
    Returns:
        float score, or None if no association found
    """
    gene_id = get_ensembl_id(gene_symbol)

    # Check if ensembl id was returned
    if gene_id is None:
        print(f"Warning: could not resolve Ensembl ID for {gene_symbol}, skipping")
        return None
    
    query_string = """
    query diseaseAssociation($efoId: String!, $targetId: [String!]) {
        disease(efoId: $efoId) {
            id
            name
            associatedTargets(Bs: $targetId) {
                rows {
                    target {
                        id
                        approvedSymbol
                    }
                    score
                }
            }
        }
    }
    """
    variables = {"efoId": disease_id, "targetId": [gene_id]}
    response = requests.post(open_targets_url, json={"query": query_string, "variables": variables})
    response.raise_for_status()
    data = response.json()
    
    disease_data = data["data"]["disease"]
    if disease_data is None:
        return None
    
    rows = disease_data["associatedTargets"]["rows"]
    for row in rows:
        if row["target"]["id"] == gene_id:
            return row["score"]
    
        return None

In [17]:
if __name__ == "__main__":
    gene = "MMP1"
    score = get_disease_target_score(disease_id, gene)
    print(score)

None


In [24]:
def build_claude_prompt(disease_name, candidate_genes, enrichment_context, string_df, kegg_df, opentargets_scores): 
    """
    Build a prompt for Claude's initial pathway hypothesis, using evidence already gathered from STRING, KEGG, and Open Targets. 

    Args: 
        disease_name: str, e.g. "hypermobile Ehlers-Danlos syndrome (hEDS)" 
        candidate_genes: list of gene symbols, e.g. ["COL5A1", "TNXB", "MMP1"]
        string_df: Dataframe from get_string_interactions()
        kegg_df: Dataframe from get_kegg_pathways_for_genes() 
        open_targets_scores: dict of {gene_symbol: score_or_None}, 
            from calling get_disease_target_score() per gene
    Returns: 
        str, the assembled prompt 
    """
    # Summarize STRING interactions 
    if string_df is not None and not string_df.empty: 
        string_summary = "\n".join(
            f"-{row['preferredName_A']} -- {row['preferredName_B']} (confidence: {row['score']:.2f})"
            for _, row in string_df.iterrows()
        )
    else: 
        string_summary = "No interactions found among candidate genes" 

    # Summarize KEGG pathway membership
    if kegg_df is not None and not kegg_df.empty: 
        kegg_summary = "\n".join(
            f"- {row['gene']}: pathway {row['pathway_id']}"
            for _, row in kegg_df.iterrows() 
        )
    else: 
        kegg_summary = "No KEGG pathway matches found among candidate genes." 

    # Summarize Open Targets disease association scores 
    ot_lines = []
    for gene in candidate_genes: 
        score = opentargets_scores.get(gene)
        if score is not None: 
            ot_lines.append(f"- {gene}: association score {score: .3f}")
        else: ot_lines.append(f"- {gene}: no existing curated association with {disease_name}")
    ot_summary = "\n".join(ot_lines) 

    prompt = f"""You are assisting with hypothesis generation for {disease_name}, a disorder without a confirmed genetic cause. Use the following evidence to identify a candidate gene pathway for disease pathogenesis. 
    Candidate genes (from GO enrichment analysis): {", ".join(candidate_genes)}
    These genes were identified together because they are enriched in: {enrichment_context}

    Evidence gathered so far: 

    STRING protein-protein interactions: 
    {string_summary} 

    KEGG pathway memberships: 
    {kegg_summary} 

    Open Targets disease association scores: 
    {ot_summary} 
    Based on this evidence, propose:
    1. A candidate mechanistic pathway describing the specific biological steps or interactions that could connect these genes to {disease_name}
    2. A connective hypothesis explaining why these particular genes, as a group, are relevant to this disease context

    Consider why these genes were grouped together by the enrichment results, and whether the STRING, KEGG, and Open Targets evidence supports a shared mechanism consistent with that enrichment signal.

    Be specific about the proposed mechanism, and explicitly note which parts of your response are supported by the evidence above versus which parts are speculative extensions beyond it.
    """
    return prompt 

In [25]:
def gather_evidence(candidate_genes, disease_name, enrichment_context): 
    """
    Run all evidence gathering functions for a set of candidate genes 
    against a given disease, and collect results into one structure. 

    Args: 
        candidate_genes: list of gene symbols e.g. ["COL5A1", "MMP1", "TNXB"] 
        disease_name: str, e.g. "hypermobile Ehlers-Danlos syndrome (hEDS)" 
        enrichment_context: str, describing the enriched GO term/pathway that 
        produced this gene set e.g. "GO:0046890 regulation of lipid biosynthetic 
        process (adj. p = 0.0003)
    Returns: 
        dict with keys: candidate_genes, disease_name, enrichment_context, string_df,
        kegg_df, opentargets_scores
    """
    # Get STRING database context 
    print(f"Gathering STRING interactions for {len(candidate_genes)} genes") 
    string_df = get_string_interactions(candidate_genes) 

    # Get KEGG pathway context 
    print(f"Gathering KEGG pathway memberships") 
    kegg_df = get_kegg_pathways(candidate_genes) 

    # Get Open Targets context 
    print(f"Gathering Open Targets scores")
    disease_id = get_disease_id(disease_name)
    if disease_id is None:
        print(f"Warning: could not resolve disease ID for '{disease_name}'")
        opentargets_scores = {gene: None for gene in candidate_genes}
    else: 
        opentargets_scores = {} 
        for gene in candidate_genes: 
            opentargets_scores[gene] = get_disease_target_score(disease_id, gene) 

    return { 
        "candidate_genes": candidate_genes, 
        "disease_name": disease_name, 
        "enrichment_context": enrichment_context, 
        "string_df": string_df, 
        "kegg_df": kegg_df, 
        "opentargets_scores": opentargets_scores
    }

    

In [26]:
if __name__ == "__main__":
    test_genes = ["APOC1", "RDH10", "APOE", "BMP6", "CES1", "RAB38", "C3"]
    evidence = gather_evidence(test_genes, "hypermobile Ehlers-Danlos syndrome", "GO:0046890 regulation of lipid biosynthetic process(padj=0.038088753)")

    prompt = build_claude_prompt(
        evidence["disease_name"], 
        evidence["candidate_genes"],
        evidence["enrichment_context"],
        evidence["string_df"], 
        evidence["kegg_df"],
        evidence["opentargets_scores"] 
    )
    print(prompt) 
    


Gathering STRING interactions for 7 genes
Gathering KEGG pathway memberships
Gathering Open Targets scores
{'id': 'MONDO_0007523', 'name': 'Ehlers-Danlos syndrome, hypermobility type', 'entity': 'disease'}
You are assisting with hypothesis generation for hypermobile Ehlers-Danlos syndrome, a disorder without a confirmed genetic cause. Use the following evidence to identify a candidate gene pathway for disease pathogenesis. 
    Candidate genes (from GO enrichment analysis): APOC1, RDH10, APOE, BMP6, CES1, RAB38, C3
    These genes were identified together because they are enriched in: GO:0046890 regulation of lipid biosynthetic process(padj=0.038088753)

    Evidence gathered so far: 

    STRING protein-protein interactions: 
    -C3 -- APOC1 (confidence: 0.43)
-C3 -- APOE (confidence: 0.73)
-APOE -- APOC1 (confidence: 1.00) 

    KEGG pathway memberships: 
    - APOC1: pathway path:hsa04979
- RDH10: pathway path:hsa00830
- RDH10: pathway path:hsa01100
- APOE: pathway path:hsa04979
- 

In [30]:
import anthropic
import os


client = anthropic.Anthropic(
    api_key=os.environ.get("ANTHROPIC_API_KEY"),
    default_headers={"anthropic-workspace-id": os.environ.get("ANTHROPIC_WORKSPACE_ID")}
)

total_input_tokens = 0
total_output_tokens = 0

INPUT_COST_PER_M = 2.00
OUTPUT_COST_PER_M = 10.00

def call_claude(prompt, max_tokens=2000, model="claude-sonnet-4-6"):
    """
    Send a prompt to Claude and track cumulative token usage/cost.
    """
    global total_input_tokens, total_output_tokens

    response = client.messages.create(
        model=model,
        max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}]
    )

    usage = response.usage
    total_input_tokens += usage.input_tokens
    total_output_tokens += usage.output_tokens

    call_cost = (usage.input_tokens / 1_000_000 * INPUT_COST_PER_M) + \
                (usage.output_tokens / 1_000_000 * OUTPUT_COST_PER_M)
    running_total_cost = (total_input_tokens / 1_000_000 * INPUT_COST_PER_M) + \
                         (total_output_tokens / 1_000_000 * OUTPUT_COST_PER_M)

    print(f"Call: {usage.input_tokens} in / {usage.output_tokens} out "
          f"(${call_cost:.4f}) | Running total: ${running_total_cost:.4f}")

    return response.content[0].text

In [31]:


if __name__ == "__main__":
    test_genes = ["APOC1", "RDH10", "APOE", "BMP6", "CES1", "RAB38", "C3"]
    evidence = gather_evidence(
        test_genes,
        "hypermobile Ehlers-Danlos syndrome",
        "GO:0046890 regulation of lipid biosynthetic process (padj=0.038088753)"
    )
    prompt = build_claude_prompt(
        disease_name=evidence["disease_name"],
        candidate_genes=evidence["candidate_genes"],
        enrichment_context=evidence["enrichment_context"],
        string_df=evidence["string_df"],
        kegg_df=evidence["kegg_df"],
        opentargets_scores=evidence["opentargets_scores"]
    )
    hypothesis = call_claude(prompt)
    print("\n--- CLAUDE'S HYPOTHESIS ---\n")
    print(hypothesis)



Gathering STRING interactions for 7 genes
Gathering KEGG pathway memberships
Gathering Open Targets scores
{'id': 'MONDO_0007523', 'name': 'Ehlers-Danlos syndrome, hypermobility type', 'entity': 'disease'}
Call: 880 in / 1826 out ($0.0200) | Running total: $0.0200

--- CLAUDE'S HYPOTHESIS ---

# Hypothesis: Lipid-Mediated Regulation of Connective Tissue Homeostasis in Hypermobile Ehlers-Danlos Syndrome

---

## 1. Candidate Mechanistic Pathway

### Step 1: Disrupted Lipid Biosynthesis Regulation (GO-Supported)
The enrichment signal (GO:0046890, *regulation of lipid biosynthetic process*) is the primary organizing principle. **APOE** and **APOC1** function together as lipid transport and metabolism regulators — supported directly by their high-confidence STRING interaction (APOE–APOC1: 1.00) and shared KEGG pathway membership (hsa04979: Cholesterol metabolism). **CES1** (carboxylesterase 1) contributes to lipid hydrolysis (KEGG: hsa00983, Drug metabolism), and **RDH10** (retinol dehydro

In [32]:
with open("heds_hypothesis.txt", "w") as f:
    f.write(prompt)
    f.write("\n\n--- CLAUDE'S HYPOTHESIS ---\n\n")
    f.write(hypothesis)